In [ ]:
from enum import StrEnum
from functools import cache
from pathlib import Path
from typing import Any, Literal

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import seaborn as sns
from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor, plot_tree

from climate_attitudes.datasets.reduced_no_imputation import schema
from climate_attitudes.visualisation import DIVERGING_CMAP, configure_mpl
from ising import Ising, SymmetricIsing

RANDOM_SEED = 202606301

rng = np.random.default_rng(RANDOM_SEED)

np.set_printoptions(linewidth=200)

configure_mpl(Path("../fonts"))

DATA_PATH = Path("../reports/thesis/results/data/model/all_interventions/")

schema = schema.post_index()

## Load data

In [ ]:
type ModelType = Literal["ising", "sym_ising"]


class InterventionStrength(StrEnum):
    NULL = "null"
    WEAK = ("weak",)
    MEDIUM = ("medium",)
    MODERATE = "moderate"
    STRONG = "strong"
    PERFECT = "perfect"

    def path(self, model_type: ModelType, data_dir: Path) -> Path:
        return data_dir / self.filename(model_type)

    def filename(self, model_type: ModelType) -> Path:
        return Path(f"{model_type}_{self.delta_str()}.npz")

    def delta(self) -> float:
        match self:
            case InterventionStrength.NULL:
                return 0.0
            case InterventionStrength.WEAK:
                return 0.5
            case InterventionStrength.MEDIUM:
                return 1.0
            case InterventionStrength.MODERATE:
                return 1.5
            case InterventionStrength.STRONG:
                return 2.5
            case InterventionStrength.PERFECT:
                return 8.0

    def delta_str(self) -> str:
        delta = self.delta()
        return str(delta).replace(".", "")


@cache
def load_intervention_results(
    strength: InterventionStrength,
    model_type: ModelType,
    data_dir: Path,
) -> np.lib.npyio.NpzFile:
    print(strength, model_type, data_dir)
    return np.load(strength.path(model_type, data_dir))


def load_intervention_effect(
    strength: InterventionStrength,
    data_dir: Path,
    measure_time: int,
    intervention_idx: int | None = None,
    target_idx: int | None = None,
    model_type: ModelType = "ising",
) -> npt.NDArray[np.int64]:
    # Load final state from intervention model
    intervention_outcome = load_intervention_results(
        strength,
        model_type,
        data_dir,
    )["measurements"][:, :, measure_time]

    # Load final state from no-intervention model
    null_outcome = load_intervention_results(
        InterventionStrength.NULL,
        model_type,
        data_dir,
    )["measurements"][:, :, measure_time]

    # Calculate effect as the counterfactual difference
    effect = intervention_outcome - null_outcome

    # Optionally filter down to a specific intervention column
    #   i.e., the belief/attitude which we influence
    if intervention_idx is not None:
        effect = effect[:, :, intervention_idx]

    # Optionally filter down to a specific target column
    #   i.e., the belief/attitude we ultimately want to change
    if target_idx is not None:
        effect = effect[..., target_idx]

    return effect


def load_effective_baseline_activation(
    strength: InterventionStrength,
    data_dir: Path,
    measure_time: int | None = None,
    intervention_idx: int | None = None,
    target_idx: int | None = None,
    model_type: ModelType = "ising",
) -> npt.NDArray[np.float64]:
    results = load_intervention_results(strength, model_type, data_dir)
    params = results["params"]
    S0 = results["Y"][:, :, 0]
    S = results["measurements"][:, :, :, intervention_idx]

    match model_type:
        case "ising":
            model_cls = Ising
        case "sym_ising":
            model_cls = SymmetricIsing
        case _:
            raise ValueError(f"Invalid model type: '{model_type}'")

    R, M, T, N = S.shape
    X = np.array([1.0])
    adj = np.ones((N, N), dtype=np.bool)
    h_eff = np.empty((R, M, T, N), dtype=np.float64)
    for repeat in range(R):
        p = params[repeat]
        h = p[:N]
        j = p[N:].reshape((N, N))

        for individual in range(M):
            for t in range(T):
                prev = S0[repeat, individual] if t == 0 else S[repeat, individual, t]
                h_eff[repeat, individual, t] = model_cls.parallel_glauber_theta(
                    prev,
                    X,
                    h,
                    j,
                    adj,
                )

    if measure_time is not None:
        h_eff = h_eff[:, :, measure_time]

    if target_idx is not None:
        h_eff = h_eff[..., target_idx]

    return h_eff

In [ ]:
null_results = load_intervention_results(InterventionStrength.NULL, "ising", DATA_PATH)

labels = null_results["labels"]

# measurements: (repeat, individual, timestep, intervention, target)
S0 = null_results["measurements"][0, :, 0, 0]

# Y0: (individual, data timestep, spin)
X0 = null_results["Y0"][:, 1]

## Compare average intervention effect to ratio of probabilities

In [ ]:
for strength in InterventionStrength:
    if strength == InterventionStrength.NULL:
        continue

    effect = load_intervention_effect(
        strength,
        DATA_PATH,
        measure_time=5,
        intervention_idx=2,
        target_idx=7,
    )

    h_eff_null = load_effective_baseline_activation(
        InterventionStrength.NULL,
        DATA_PATH,
        measure_time=5,
        intervention_idx=2,
        target_idx=7,
    )

    h_eff_int = load_effective_baseline_activation(
        strength,
        DATA_PATH,
        measure_time=5,
        intervention_idx=2,
        target_idx=7,
    )

    avg_p_null = (np.exp(h_eff_null) / (2 * np.cosh(h_eff_null))).mean(axis=0)
    avg_p_int = (np.exp(h_eff_int) / (2 * np.cosh(h_eff_int))).mean(axis=0)
    avg_p_ratio = avg_p_int / avg_p_null
    avg_effect = effect.mean(axis=0)

    plt.scatter(avg_effect, avg_p_ratio)
    plt.xlabel("Effect of intervention")
    plt.ylabel("Ratio of $P(S_i^5 = +1)$")
    plt.title(f"Intervention: {strength.delta()}")

## Regressing average intervention effect on initial state

For each individual, calculate the average effect of intervention ($S_{i, \text{intervention}}^t - S_{i, \text{null}}^t$), and fit a linear regression model for this, taking the initial raw measurement as a predictor. Include pairwise polynomial features.

In [ ]:
def fit_lasso_cv(X, Y) -> tuple[npt.NDArray[np.float64], npt.NDArray[np.float64], Any]:
    model = Pipeline([("poly", PolynomialFeatures()), ("lasso", LassoCV(cv=10))]).fit(
        X, Y
    )
    degree_1_features = model["lasso"].coef_[1:9]
    degree_2_features = np.zeros((8, 8), dtype=np.float64)
    degree_2_features[np.triu_indices_from(degree_2_features)] = model["lasso"].coef_[
        9:
    ]

    degree_1_features[abs(degree_1_features) < 1e-4] = 0
    degree_2_features[abs(degree_2_features) < 1e-4] = 0

    return degree_1_features, degree_2_features, model

In [ ]:
for intervention in (
    InterventionStrength.WEAK,
    InterventionStrength.MODERATE,
    InterventionStrength.STRONG,
):
    if intervention == InterventionStrength.NULL:
        continue

    # poly = PolynomialFeatures()
    # X = poly.fit_transform(X0)
    Y = load_intervention_effect(
        intervention, DATA_PATH, measure_time=5, intervention_idx=2, target_idx=7
    ).mean(axis=0)
    β1, β2, model = fit_lasso_cv(X0, Y)

    fig, axes = plt.subplots(
        nrows=2, figsize=(5.5, 6.5), constrained_layout=True, height_ratios=(1, 3)
    )
    sns.barplot(β1, ax=axes[0])
    axes[0].set_xticks(np.arange(8), labels, rotation=45, horizontalalignment="right")

    axes[1].set_aspect("equal")
    mask = np.triu(np.ones((8, 8), dtype=np.bool), k=1)[:, ::-1]
    sns.heatmap(
        β2[::-1],
        mask=mask,
        vmin=-0.05,
        vmax=0.05,
        center=0,
        cmap=DIVERGING_CMAP,
        annot=True,
        fmt=".2f",
        linewidth=0.5,
        cbar_kws=dict(shrink=0.65, aspect=25),
        ax=axes[1],
    )
    axes[1].set_yticks(np.arange(8) + 0.5, reversed(labels), rotation=0)
    axes[1].set_xticks(
        np.arange(8) + 0.5, labels, rotation=35, horizontalalignment="right"
    )

    fig.suptitle(f"$\\delta = {intervention.delta()}$")
    plt.show()

    fig, ax = plt.subplots(figsize=(5.5, 5.5), constrained_layout=True)
    ax.set_aspect("equal")
    Y_pred = model.predict(X0)
    ax.scatter(
        Y,
        Y_pred,
        s=10,
    )
    ax.set_xlabel(r"$Y$")
    ax.set_ylabel(r"$\hat{Y}$")
    fig.suptitle(f"$\\delta = {intervention.delta()}$")
    plt.show()

## Regressing average intervention effect on initial local fields

For each individual, calculate the average effect of intervention ($S_{i, \text{intervention}}^t - S_{i, \text{null}}^t$), and fit a linear regression model for this, taking the average initial _local fields_ as a predictor. Include pairwise polynomial features.

In [ ]:
h_eff = load_effective_baseline_activation(
    InterventionStrength.NULL,
    DATA_PATH,
    measure_time=0,
    intervention_idx=2,
).mean(axis=0)

for intervention in InterventionStrength:
    if intervention == InterventionStrength.NULL:
        continue

    poly = PolynomialFeatures()
    X = poly.fit_transform(h_eff)
    Y = load_intervention_effect(
        intervention,
        DATA_PATH,
        measure_time=5,
        intervention_idx=2,
        target_idx=7,
    ).mean(axis=0)
    β1, β2 = fit_lasso_cv(X, Y)

    fig, axes = plt.subplots(
        nrows=2,
        figsize=(5.5, 6.5),
        constrained_layout=True,
        height_ratios=(1, 3),
    )
    sns.barplot(β1, ax=axes[0])
    axes[0].set_xticks(np.arange(8), labels, rotation=45, horizontalalignment="right")

    axes[1].set_aspect("equal")
    mask = np.triu(np.ones((8, 8), dtype=np.bool), k=1)[:, ::-1]
    sns.heatmap(
        β2[::-1],
        mask=mask,
        vmin=-0.05,
        vmax=0.05,
        center=0,
        cmap=DIVERGING_CMAP,
        annot=True,
        fmt=".2f",
        linewidth=0.5,
        cbar_kws=dict(shrink=0.65, aspect=25),
        ax=axes[1],
    )
    axes[1].set_yticks(
        np.arange(8) + 0.5,
        reversed(labels),
        rotation=0,
    )
    axes[1].set_xticks(
        np.arange(8) + 0.5,
        labels,
        rotation=35,
        horizontalalignment="right",
    )

    fig.suptitle(f"$\\delta = {intervention.delta()}$")

## Regressing ratio of final probabilities on initial activation probabilities

For each individual, calculate the average probability that the target spin is $+1$ at $t=5$, and the corresponding ratio with the average probability in the null model. Fit a linear regresson model to this, taking the average initial spin probabilities as predictors. 

In [ ]:
h_eff = load_effective_baseline_activation(
    InterventionStrength.NULL,
    DATA_PATH,
    measure_time=0,
    intervention_idx=2,
).mean(axis=0)

init_p = np.exp(h_eff) / (2 * np.cosh(h_eff))

for intervention in (
    InterventionStrength.WEAK,
    InterventionStrength.MODERATE,
    InterventionStrength.STRONG,
):
    if intervention == InterventionStrength.NULL:
        continue

    h_eff_final_null = load_effective_baseline_activation(
        InterventionStrength.NULL,
        DATA_PATH,
        measure_time=5,
        intervention_idx=2,
        target_idx=7,
    ).mean(axis=0)
    h_eff_final_int = load_effective_baseline_activation(
        intervention,
        DATA_PATH,
        measure_time=5,
        intervention_idx=2,
        target_idx=7,
    ).mean(axis=0)

    final_p_null = np.exp(h_eff_final_null) / (2 * np.cosh(h_eff_final_null))
    final_p_int = np.exp(h_eff_final_int) / (2 * np.cosh(h_eff_final_int))

    ratio = np.exp(np.log(final_p_int) - np.log(final_p_null))

    X = init_p
    Y = ratio
    model = LassoCV(cv=10).fit(X, Y)
    β1 = model.coef_

    fig, ax = plt.subplots(
        figsize=(5.5, 2.5),
        constrained_layout=True,
    )
    sns.barplot(β1, ax=ax)
    ax.set_xticks(np.arange(8), labels, rotation=45, horizontalalignment="right")
    ax.set_ylim(-1.0, 0.2)
    ax.axhline(y=0, linestyle="dashed", linewidth=0.5, color="grey")

    fig.suptitle(f"$\\delta = {intervention.delta()}$")
    plt.show()

## Regressing ratio of final probabilities on initial state

For each individual, calculate the average probability that the target spin is $+1$ at $t=5$, and the corresponding ratio with the average probability in the null model. Fit a linear regresson model to this, taking the initial raw measurement as a predictor. Include pairwise polynomial features.

In [ ]:
for intervention in (
    InterventionStrength.WEAK,
    InterventionStrength.MODERATE,
    InterventionStrength.STRONG,
):
    if intervention == InterventionStrength.NULL:
        continue

    h_eff_final_null = load_effective_baseline_activation(
        InterventionStrength.NULL,
        DATA_PATH,
        measure_time=5,
        intervention_idx=2,
        target_idx=7,
    ).mean(axis=0)
    h_eff_final_int = load_effective_baseline_activation(
        intervention,
        DATA_PATH,
        measure_time=5,
        intervention_idx=2,
        target_idx=7,
    ).mean(axis=0)

    final_p_null = np.exp(h_eff_final_null) / (2 * np.cosh(h_eff_final_null))
    final_p_int = np.exp(h_eff_final_int) / (2 * np.cosh(h_eff_final_int))

    ratio = np.exp(np.log(final_p_int) - np.log(final_p_null))

    # poly = PolynomialFeatures()
    # X = poly.fit_transform(X0)
    Y = ratio
    β1, β2, model = fit_lasso_cv(X0, Y)

    fig, axes = plt.subplots(
        nrows=2,
        figsize=(5.5, 6.5),
        constrained_layout=True,
        height_ratios=(1, 3),
    )
    sns.barplot(β1, ax=axes[0])
    axes[0].set_xticks(np.arange(8), labels, rotation=45, horizontalalignment="right")

    axes[1].set_aspect("equal")
    mask = np.triu(np.ones((8, 8), dtype=np.bool), k=1)[:, ::-1]
    vlim = np.max(abs(β2))
    sns.heatmap(
        β2[::-1],
        mask=mask,
        vmin=-vlim,
        vmax=vlim,
        center=0,
        cmap=DIVERGING_CMAP,
        annot=True,
        fmt=".2f",
        linewidth=0.5,
        cbar_kws=dict(shrink=0.65, aspect=25),
        ax=axes[1],
    )
    axes[1].set_yticks(
        np.arange(8) + 0.5,
        reversed(labels),
        rotation=0,
    )
    axes[1].set_xticks(
        np.arange(8) + 0.5,
        labels,
        rotation=35,
        horizontalalignment="right",
    )

    fig.suptitle(f"$\\delta = {intervention.delta()}$")
    plt.show()

    fig, ax = plt.subplots(figsize=(5.5, 5.5), constrained_layout=True)
    ax.set_aspect("equal")
    Y_pred = model.predict(X0)
    ax.scatter(
        Y,
        Y_pred,
        s=10,
    )
    ax.set_xlabel(r"$Y$")
    ax.set_ylabel(r"$\hat{Y}$")
    fig.suptitle(f"$\\delta = {intervention.delta()}$")
    plt.show()

## Shallow decision tree

In [ ]:
from dataclasses import dataclass
from queue import deque


@dataclass
class RuleNode:
    feature_idx: int
    le: bool
    value: float


@dataclass
class Rule:
    nodes: list[RuleNode]

    def print(self, labels: list[str]):
        print(
            " && ".join(
                (
                    f"({labels[node.feature_idx]} {'<=' if node.le else '>'} "
                    f"{node.value:.2f})"
                )
                for node in self.nodes
            )
        )


@dataclass
class Rules:
    rules: list[Rule]
    le: float | None = None
    ge: float | None = None

    @classmethod
    def extract(cls, tree_clf, le: float | None = None, ge: float | None = None):
        rules = extract_rules(tree_clf, le, ge)
        return cls(
            le=le,
            ge=ge,
            rules=rules,
        )

    def print(self, labels: list[str]):
        for rule in self.rules:
            rule.print(labels)


def extract_rules(tree_clf, le: float | None = None, ge: float | None = None):
    _le = le or np.inf
    _ge = ge or -np.inf

    # Get list of paths (index tuples) which lead to desired predictions
    paths = []
    queue = deque()
    queue.append((0,))
    while queue:
        path = queue.popleft()
        head = path[-1]
        children = [
            tree_clf.tree_.children_left[head],
            tree_clf.tree_.children_right[head],
        ]
        is_leaf = True
        for c in children:
            if c == -1:
                continue
            queue.append(path + (int(c),))
            is_leaf = False

        if is_leaf and _ge <= tree_clf.tree_.value[head] <= _le:
            paths.append(path)

    # Convert each path to a rule:
    rules = []
    for path in paths:
        rule = []
        for i in range(len(path) - 1):
            rule.append(
                RuleNode(
                    feature_idx=int(tree_clf.tree_.feature[path[i]]),
                    le=bool(path[i + 1] == tree_clf.tree_.children_left[path[i]]),
                    value=tree_clf.tree_.threshold[path[i]].item(),
                )
            )
        rules.append(Rule(nodes=rule))

    return rules

In [ ]:
# from dataclasses import dataclass
# from queue import deque

# @dataclass
# class FeatureBounds:
#     lower: float = -np.inf
#     upper: float = np.inf


# def extract_rules(tree_clf, le: float | None = None, ge: float | None = None):
#     _le = le or np.inf
#     _ge = ge or -np.inf

#     # Get list of paths (index tuples) which lead to desired predictions
#     paths = []
#     queue = deque()
#     queue.append((0,))
#     while queue:
#         path = queue.popleft()
#         head = path[-1]
#         children = [
# tree_clf.tree_.children_left[head],
# tree_clf.tree_.children_right[head]]
#         is_leaf = True
#         for c in children:
#             if c == -1:
#                 continue
#             queue.append(path + (int(c),))
#             is_leaf = False

#         if is_leaf and _ge <= tree_clf.tree_.value[head] <= _le:
#             paths.append(path)

#     # Convert each path to a rule:
#     rules = []
#     for path in paths:
#         bounds = {}

#         for i in range(len(path) - 1):
#             feature_idx = int(tree_clf.tree_.feature[path[i]])
#             b = bounds.setdefault(feature_idx, FeatureBounds())
#             threshold = tree_clf.tree_.threshold[path[i]].item()
#             is_left_child = path[i+1] == tree_clf.tree_.children_left[path[i]]
#             if is_left_child:
#                 b.upper = min(b.upper, threshold)
#             else:
#                 b.lower = max(b.lower, threshold)
#         rules.append(bounds)

#     return rules

In [ ]:
rules = extract_rules(model, ge=1.5)

In [ ]:
# did_merge = True
# while did_merge:
#     merge_pair = None
#     merge_on = None
#     for i in range(len(rules)):
#         for j in range(len(rules)):
#             if i == j:
#                 continue
#             if rules[i].keys() != rules[j].keys():
#                 continue

#             merge_pair = (i,j)
#             merge_on = None
#             for feature_idx in rules[i].keys():
#                 if rules[i][feature_idx] == rules[j][feature_idx]:
#                     continue
#                 # elif (
# rules[i][feature_idx].lower != rules[j][feature_idx].lower
# and rules[i][feature_idx].upper != rules[j][feature_idx].upper
# ):
#                 #     print(i, j)
#                 #     print(feature_idx)
#                 #     print(rules[i][feature_idx], rules[j][feature_idx])
#                 #     break
#                 if merge_on is not None:
#                     merge_pair = None
#                     break
#                 merge_on = feature_idx
#             if merge_on is not None and merge_pair is not None:
#                 break
#         if merge_on is not None and merge_pair is not None:
#             break

#     if merge_on is not None and merge_pair is not None:
#         rules[i][merge_on].lower = min(
# rules[i][merge_on].lower,
# rules[j][merge_on].lower,
# )
#         rules[i][merge_on].upper = max(
# rules[i][merge_on].upper,
# rules[j][merge_on].upper,
# )
#         print(j)
#         rules.pop(j)
#         print("Hi")
#     else:
#         did_merge = False

# for i, rule in enumerate(rules):
#     remove_idxs = []
#     for idx, bounds in rule.items():
#         if bounds.lower == -np.inf and bounds.upper == np.inf:
#             remove_idxs.append(idx)
#     rules[i] = {
# _idx: bounds
# for _idx, bounds in rule.items()
# if _idx not in remove_idxs
# }

In [ ]:
model = DecisionTreeRegressor(max_depth=4)
model.fit(X0, Y)
plot_tree(model);

Effective when:
- Low existing support for climate policies, and either skepticism about human causes of climate change, or low existing worry about climate change
- Neutral--positive support for climate policies, but low worry and republican views

In [ ]:
rules = Rules.extract(model, ge=1.5)
rules.print(labels)

Ineffective for individuals who already support climate policy, and who are already worried about climate change.

In [ ]:
rules = Rules.extract(model, le=1.2)
rules.print(labels)

## Comparing regression and tree models

In [ ]:
from sklearn.model_selection import RepeatedKFold, cross_validate

$R^2$ score

In [ ]:
lasso = Pipeline([("poly", PolynomialFeatures()), ("lasso", LassoCV(cv=10))])
tree = DecisionTreeRegressor(max_depth=4)

cv = RepeatedKFold(n_splits=10, n_repeats=5, random_state=RANDOM_SEED)

lasso_scores = cross_validate(lasso, X0, Y, cv=cv, scoring="r2")

tree_scores = cross_validate(tree, X0, Y, cv=cv, scoring="r2")

In [ ]:
(
    lasso_scores["test_score"].mean(),
    tree_scores["test_score"].mean(),
)  # , rf_scores["test_score"].mean()

How well does each model recover the top-percentile (most effective) intervention individuals?

In [ ]:
true_top_10_pct = np.argwhere(np.percentile(Y, q=90) <= Y).flatten()

lasso_pred = lasso.fit(X, Y).predict(X)
tree_pred = tree.fit(X, Y).predict(X)

lasso_top_10_pct = np.argwhere(lasso_pred >= np.percentile(lasso_pred, q=90)).flatten()
tree_top_10_pct = np.argwhere(tree_pred >= np.percentile(tree_pred, q=90)).flatten()

In [ ]:
# How many true top-percentile respondents are recovered?
recovery_lasso = (
    np.intersect1d(true_top_10_pct, lasso_top_10_pct).size / true_top_10_pct.size
)
recovery_tree = (
    np.intersect1d(true_top_10_pct, tree_top_10_pct).size / true_top_10_pct.size
)

In [ ]:
recovery_lasso, recovery_tree